# DAY 3 v2 — Traditional ML Baseline: Vietnamese Price Prediction

**Dataset:** `SeanSunny/items_tv_v9` (train=269,112 | val=3,926 | test=3,872)  
**Price unit:** `price = round(price_vnd / 1000)` — range 5–1000 (nghìn VND, same scale as English)  
**Primary metric:** MAE (k VND) — aligned with English pipeline  
**Training target:** raw `item.price` (5–1000), NO log transform — same as English day3  

## Nội dung notebook này (Section 0–4)

| Section | Nội dung |
|---|---|
| 0 | Setup & Load data |
| 1 | Statistical baselines (random, mean, median, category_mean) |
| 2 | Linear Regression baselines (simple features, BoW, TF-IDF char_wb) |
| 3 | Benchmark vectorizer: BoW vs char_wb vs Underthesea (50K subset + LGB default) |
| 4 | RandomForest + XGBoost + bảng so sánh tổng hợp |

## Section 0 — Setup & Load Data

In [ ]:
import random
import sys
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.ensemble import RandomForestRegressor
import lightgbm as lgb
import xgboost as xgb

sys.path.insert(0, "..")
from pricer_vi_2.items import Item
from pricer_vi_2.evaluator import evaluate

In [ ]:
train, val, test = Item.from_hub("SeanSunny/items_tv_v9")
print(f"Loaded {len(train):,} train | {len(val):,} val | {len(test):,} test")

In [ ]:
# EDA nhanh — hiểu phân phối giá trước khi model
train_prices = np.array([item.price for item in train])
print(f"Price range : {train_prices.min():.0f} – {train_prices.max():.0f} k VND")
print(f"Mean        : {train_prices.mean():.1f} k VND")
print(f"Median      : {np.median(train_prices):.1f} k VND")
print(f"Std         : {train_prices.std():.1f} k VND")
print()

print("Category distribution (train):")
for cat, cnt in Counter(item.category for item in train).most_common():
    print(f"  {cat:35s}: {cnt:,} ({cnt/len(train)*100:.1f}%)")
print()

summary_lens = [len(item.summary) for item in train]
print(f"Summary length: mean={np.mean(summary_lens):.0f} | min={min(summary_lens)} | max={max(summary_lens)}")
print()

print("Sample item:")
item0 = train[0]
print(f"  title   : {item0.title}")
print(f"  category: {item0.category}")
print(f"  brand   : {item0.brand}")
print(f"  price   : {item0.price} (k VND)")
print(f"  summary :\n{item0.summary}")

In [ ]:
# Dict chứa kết quả tất cả models — dùng để so sánh cuối Section 4
results = {}

## Section 1 — Statistical Baselines

Không dùng NLP — chỉ là heuristic đơn giản.  
Đây là "đường sàn" mà mọi ML model đều phải vượt qua.

Evaluate trên `test` set, 200 samples — giống English day3.

In [ ]:
def random_pricer(item):
    return random.randrange(5, 1001)

random.seed(42)
results["1a. Random"] = evaluate(random_pricer, test)

In [ ]:
training_average = sum(item.price for item in train) / len(train)
print(f"Training average price: {training_average:.1f} k VND")

def constant_pricer(item):
    return training_average

results["1b. Constant (Mean)"] = evaluate(constant_pricer, test)

In [ ]:
training_median = float(np.median(train_prices))
print(f"Training median price: {training_median:.1f} k VND")

def median_pricer(item):
    return training_median

results["1c. Median"] = evaluate(median_pricer, test)

In [ ]:
cat_price_sums = defaultdict(float)
cat_counts_map = defaultdict(int)
for item in train:
    cat_price_sums[item.category] += item.price
    cat_counts_map[item.category] += 1
cat_means = {cat: cat_price_sums[cat] / cat_counts_map[cat] for cat in cat_price_sums}

print("Category mean prices (k VND):")
for cat, mean_p in sorted(cat_means.items(), key=lambda x: x[1]):
    print(f"  {cat:35s}: {mean_p:.1f}")

def category_mean_pricer(item):
    return cat_means.get(item.category, training_average)

results["1d. Category Mean"] = evaluate(category_mean_pricer, test)

## Section 2 — Linear Regression Baselines

Train trực tiếp trên raw price (5–1000) — giống English day3.  
Không dùng log transform vì price range đã align với English.

In [ ]:
# Prepare targets và documents
prices = np.array([float(item.price) for item in train])
documents = [item.summary for item in train]
print(f"Train prices: {len(prices):,} items, range {prices.min():.0f}–{prices.max():.0f}")

In [ ]:
# 2a. LR với features đơn giản: text_length, title_length
# English dùng weight + text_length; ta không có weight nên dùng text_length + title_length

def get_features(item):
    return {
        "text_length": len(item.summary),
        "title_length": len(item.title),
    }

def list_to_df(items):
    feats = [get_features(item) for item in items]
    df = pd.DataFrame(feats)
    df["price"] = [item.price for item in items]
    return df

train_df = list_to_df(train)
test_df = list_to_df(test)

feat_cols = ["text_length", "title_length"]

np.random.seed(42)
lr_simple = LinearRegression()
lr_simple.fit(train_df[feat_cols], train_df["price"])

for feat, coef in zip(feat_cols, lr_simple.coef_):
    print(f"  {feat}: {coef:.4f}")
print(f"  Intercept: {lr_simple.intercept_:.4f}")

mse = mean_squared_error(test_df["price"], lr_simple.predict(test_df[feat_cols]))
r2 = r2_score(test_df["price"], lr_simple.predict(test_df[feat_cols]))
print(f"  MSE: {mse:,.0f}  R²: {r2:.4f}")

def linear_regression_pricer(item):
    feats = pd.DataFrame([get_features(item)])
    return max(5, lr_simple.predict(feats)[0])

results["2a. LR Simple Features"] = evaluate(linear_regression_pricer, test)

In [ ]:
# 2b. LR + CountVectorizer (Bag-of-Words, 2000 features) — giống English day3

np.random.seed(42)
vec_bow = CountVectorizer(max_features=2000)
X_bow = vec_bow.fit_transform(documents)
print(f"BoW vocabulary size: {len(vec_bow.get_feature_names_out())}")
print(f"Sample words: {vec_bow.get_feature_names_out()[1000:1010]}")

lr_bow = LinearRegression()
lr_bow.fit(X_bow, prices)

def natural_language_linear_regression_pricer(item):
    x = vec_bow.transform([item.summary])
    return max(5, lr_bow.predict(x)[0])

results["2b. LR + BoW"] = evaluate(natural_language_linear_regression_pricer, test)

In [ ]:
# 2c. LR + TF-IDF char_wb (2-4gram, 100K vocab)
# Vectorizer chính cho tiếng Việt — không cần word segmentation

vec_tfidf = TfidfVectorizer(
    analyzer="char_wb",
    ngram_range=(2, 4),
    max_features=100_000,
    sublinear_tf=True,
)
X_tfidf = vec_tfidf.fit_transform(documents)
print(f"TF-IDF char_wb features: {X_tfidf.shape}")

lr_tfidf = LinearRegression()
lr_tfidf.fit(X_tfidf, prices)

def tfidf_lr_pricer(item):
    x = vec_tfidf.transform([item.summary])
    return max(5, lr_tfidf.predict(x)[0])

results["2c. LR + TF-IDF char_wb"] = evaluate(tfidf_lr_pricer, test)

## Section 3 — Benchmark Vectorizer

So sánh 3 vectorizer với LGB default trên **50K subset** để chọn vectorizer tốt nhất cho Section 4.

| # | Vectorizer | Ghi chú |
|---|---|---|
| A | `CountVectorizer` — Bag-of-Words | Baseline nhanh nhất |
| B | `TfidfVectorizer(char_wb, 2-4gram)` | **Khuyến nghị chính** |
| C | `Underthesea + TfidfVectorizer(word)` | Chậm (~20-30 phút cho 50K) |

In [ ]:
# Cài underthesea nếu chưa có
import subprocess
subprocess.run(["uv", "add", "underthesea"], check=True)

In [ ]:
# 50K subset cho benchmark
BENCH_SIZE = 50_000
train_sub = train[:BENCH_SIZE]
docs_sub = [item.summary for item in train_sub]
prices_sub = np.array([float(item.price) for item in train_sub])
print(f"Benchmark subset: {BENCH_SIZE:,} items, price range {prices_sub.min():.0f}–{prices_sub.max():.0f}")

In [ ]:
# 3A. BoW + LGB default
print("[3A] BoW + LGB...")
vec_bow_b = CountVectorizer(max_features=2000)
X_bow_b = vec_bow_b.fit_transform(docs_sub)

lgb_bow = lgb.LGBMRegressor(
    n_estimators=500, num_leaves=31, learning_rate=0.1,
    n_jobs=-1, random_state=42, verbose=-1,
)
lgb_bow.fit(X_bow_b, prices_sub)

def bow_lgb_bench(item):
    x = vec_bow_b.transform([item.summary])
    return max(5, lgb_bow.predict(x)[0])

print("Evaluating [3A] BoW + LGB (200 test samples):")
results["3A. Bench BoW + LGB"] = evaluate(bow_lgb_bench, test)

In [ ]:
# 3B. char_wb TF-IDF + LGB default
print("[3B] char_wb TF-IDF + LGB...")
vec_cw_b = TfidfVectorizer(
    analyzer="char_wb", ngram_range=(2, 4),
    max_features=100_000, sublinear_tf=True,
)
X_cw_b = vec_cw_b.fit_transform(docs_sub)
print(f"  Feature matrix: {X_cw_b.shape}")

lgb_cw = lgb.LGBMRegressor(
    n_estimators=500, num_leaves=31, learning_rate=0.1,
    n_jobs=-1, random_state=42, verbose=-1,
)
lgb_cw.fit(X_cw_b, prices_sub)

def charwb_lgb_bench(item):
    x = vec_cw_b.transform([item.summary])
    return max(5, lgb_cw.predict(x)[0])

print("Evaluating [3B] char_wb + LGB (200 test samples):")
results["3B. Bench char_wb + LGB"] = evaluate(charwb_lgb_bench, test)

In [ ]:
# 3C. Underthesea + TF-IDF word + LGB default
# LƯU Ý: tokenize 50K docs mất khoảng 20-30 phút
from underthesea import word_tokenize

def vi_tokenize(text):
    return word_tokenize(text, format="text")

print("[3C] Underthesea tokenizing 50K docs (~20-30 min)...")
docs_sub_vi = [vi_tokenize(doc) for doc in docs_sub]
print("Tokenization done.")

vec_vi_b = TfidfVectorizer(max_features=100_000, sublinear_tf=True)
X_vi_b = vec_vi_b.fit_transform(docs_sub_vi)
print(f"  Feature matrix: {X_vi_b.shape}")

lgb_vi = lgb.LGBMRegressor(
    n_estimators=500, num_leaves=31, learning_rate=0.1,
    n_jobs=-1, random_state=42, verbose=-1,
)
lgb_vi.fit(X_vi_b, prices_sub)

def under_lgb_bench(item):
    tok = vi_tokenize(item.summary)
    x = vec_vi_b.transform([tok])
    return max(5, lgb_vi.predict(x)[0])

print("Evaluating [3C] Underthesea + LGB (200 test samples):")
results["3C. Bench Underthesea + LGB"] = evaluate(under_lgb_bench, test)

In [ ]:
# Bảng so sánh 3 vectorizer
print("=" * 65)
print("VECTORIZER BENCHMARK (50K train subset, LGB default, 200 test)")
print("=" * 65)
bench_keys = ["3A. Bench BoW + LGB", "3B. Bench char_wb + LGB", "3C. Bench Underthesea + LGB"]
for name in bench_keys:
    m = results[name]
    print(f"{name:35s}  MAE={m['mae']:.1f}k  MSE={m['mse']:,.0f}  R²={m['r2']:.1f}%")
print("=" * 65)
best_vec = min(bench_keys, key=lambda k: results[k]["mae"])
print(f"Best vectorizer (MAE): {best_vec}")

## Section 4 — RandomForest & XGBoost

Dùng vectorizer tốt nhất từ Section 3 (kỳ vọng: `char_wb`), refit trên **toàn bộ train** (269K).

- **RandomForest**: 15K subset (slow với 269K) — giống English day3  
- **XGBoost**: full 269K — giống English day3 (có thể mất 30-60 phút)

In [ ]:
# Refit char_wb vectorizer trên full 269K train
# Đổi sang BoW nếu Section 3 benchmark cho thấy BoW tốt hơn
print("Fitting char_wb TF-IDF on all 269K training docs...")
vec_main = TfidfVectorizer(
    analyzer="char_wb",
    ngram_range=(2, 4),
    max_features=100_000,
    sublinear_tf=True,
)
X_main = vec_main.fit_transform(documents)
print(f"Feature matrix: {X_main.shape}")

In [ ]:
# 4a. Random Forest — 15K subset, giống English day3
subset = 15_000
rf_model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=4)
rf_model.fit(X_main[:subset], prices[:subset])

def random_forest(item):
    x = vec_main.transform([item.summary])
    return max(5, rf_model.predict(x)[0])

results["4a. RandomForest (15K)"] = evaluate(random_forest, test)

In [ ]:
# 4b. XGBoost — full 269K train, giống English day3
# LƯU Ý: tree_method='hist' tối ưu cho sparse matrix, có thể mất 30-60 phút
np.random.seed(42)
xgb_model = xgb.XGBRegressor(
    n_estimators=1000,
    random_state=42,
    n_jobs=4,
    learning_rate=0.1,
    tree_method="hist",
)
xgb_model.fit(X_main, prices)

def xg_boost(item):
    x = vec_main.transform([item.summary])
    return max(5, xgb_model.predict(x)[0])

results["4b. XGBoost"] = evaluate(xg_boost, test)

In [ ]:
# === BẢNG SO SÁNH TỔNG HỢP — Section 0–4 ===
print("=" * 70)
print("LEADERBOARD — Day 3 v2 (Section 0-4, 200 test samples)")
print("=" * 70)
print(f"{'Model':35s}  {'MAE (k VND)':>12}  {'MSE':>10}  {'R²':>7}")
print("-" * 70)
for name, m in results.items():
    print(f"{name:35s}  {m['mae']:>9.1f}k     {m['mse']:>10,.0f}  {m['r2']:>6.1f}%")
print("=" * 70)
best = min(results, key=lambda k: results[k]["mae"])
print(f"Best so far: {best}  MAE={results[best]['mae']:.1f}k VND")
print()
print("Day 3 v1 tham chieu: MAE ~110k VND")